In [10]:
import numpy as np
import torch
import plotly.express as px
import plotly.graph_objects as go

In [11]:
knots_u = torch.tensor([0.0, 0.0, 0.0, 0.5, 0.5, 1.0, 1.0, 1.0])
knots_v = torch.tensor([0.0, 0.0, 0.0, 0.25, 0.25, 0.5, 0.5, 0.75, 0.75, 1.0, 1.0, 1.0])

control_points_flat = torch.tensor([
    [0.0000, 0.0000, -1.0000, 1.0000],
    [-1.0000, 0.0000, -1.0000, 0.7071],
    [-1.0000, 0.0000, 0.0000, 1.0000],
    [-1.0000, 0.0000, 1.0000, 0.7071],
    [0.0000, 0.0000, 1.0000, 1.0000],
    [0.0000, 0.0000, -1.0000, 0.7071],
    [-1.0000, -1.0000, -1.0000, 0.5000],
    [-1.0000, -1.0000, 0.0000, 0.7071],
    [-1.0000, -1.0000, 1.0000, 0.5000],
    [0.0000, 0.0000, 1.0000, 0.7071],
    [0.0000, 0.0000, -1.0000, 1.0000],
    [0.0000, -1.0000, -1.0000, 0.7071],
    [0.0000, -1.0000, 0.0000, 1.0000],
    [0.0000, -1.0000, 1.0000, 0.7071],
    [0.0000, 0.0000, 1.0000, 1.0000],
    [0.0000, 0.0000, -1.0000, 0.7071],
    [1.0000, -1.0000, -1.0000, 0.5000],
    [1.0000, -1.0000, 0.0000, 0.7071],
    [1.0000, -1.0000, 1.0000, 0.5000],
    [0.0000, 0.0000, 1.0000, 0.7071],
    [0.0000, 0.0000, -1.0000, 1.0000],
    [1.0000, 0.0000, -1.0000, 0.7071],
    [1.0000, 0.0000, 0.0000, 1.0000],
    [1.0000, 0.0000, 1.0000, 0.7071],
    [0.0000, 0.0000, 1.0000, 1.0000],
    [0.0000, 0.0000, -1.0000, 0.7071],
    [1.0000, 1.0000, -1.0000, 0.5000],
    [1.0000, 1.0000, 0.0000, 0.7071],
    [1.0000, 1.0000, 1.0000, 0.5000],
    [0.0000, 0.0000, 1.0000, 0.7071],
    [0.0000, 0.0000, -1.0000, 1.0000],
    [0.0000, 1.0000, -1.0000, 0.7071],
    [0.0000, 1.0000, 0.0000, 1.0000],
    [0.0000, 1.0000, 1.0000, 0.7071],
    [0.0000, 0.0000, 1.0000, 1.0000],
    [0.0000, 0.0000, -1.0000, 0.7071],
    [-1.0000, 1.0000, -1.0000, 0.5000],
    [-1.0000, 1.0000, 0.0000, 0.7071],
    [-1.0000, 1.0000, 1.0000, 0.5000],
    [0.0000, 0.0000, 1.0000, 0.7071],
])

In [12]:
class NURBS:

    def __init__(self, control_points, vector_knot_u, vector_knot_v, degree = 3, cycle = True):
        self.control_points = control_points
        self.vector_knot_u = vector_knot_u
        self.vector_knot_v = vector_knot_v
        self.degree = degree
        self.cycle = cycle


    def cox_de_boor(self, discretisation, knots):

        def divide(numerator, denominator, thresh = 1e-6):
            return torch.where(denominator.abs() < thresh, torch.zeros_like(numerator), numerator / denominator)

        discretisation = discretisation.unsqueeze(-1)
        lower_bound = knots[..., :-1]
        upper_bound = knots[..., 1:]

        epsilon = 1e-6
        max_val = knots.max() - epsilon
        clamped = torch.min(discretisation, max_val)

        b = ((clamped >= lower_bound) & (clamped < upper_bound)).float()

        for d in range(1, self.degree + 1):
            left = divide((discretisation - knots[..., :-(d + 1)]), (knots[..., d:-1] - knots[..., :-(d + 1)]))
            right = divide((knots[..., (d + 1):] - discretisation), (knots[..., (d + 1):] - knots[..., 1: -d]))

            b = left * b[..., :-1] + right * b[..., 1:]

        return b

    def tessellate(self):
        weighted_control_points = torch.cat([self.control_points, self.control_points[ :self.degree - 1]])
        weighted_control_points[..., :3] = weighted_control_points[..., :3] * weighted_control_points[..., 3:4]

        resolution_u = weighted_control_points.shape[1] * 10
        resolution_v = weighted_control_points.shape[0] * 10

        b_u = self.cox_de_boor(torch.linspace(0, 1, resolution_u), self.vector_knot_u)
        b_v = self.cox_de_boor(torch.linspace(0, 1, resolution_v), self.vector_knot_v)

        coordinate_4d = torch.einsum('iu, jv, vuw -> ijw', b_u, b_v, weighted_control_points)

        coordinate_3d = coordinate_4d[..., :3] / coordinate_4d[..., 3:4]
        coordinate_3d = coordinate_3d.reshape(-1, 3)

        triangles = []

        def idx(u, v):
            return u * resolution_v + v

        for u in range(resolution_u - 1):
            for v in range(resolution_v - 1):
                b_l = idx(u, v)
                b_r = idx(u, v + 1)
                t_l = idx(u + 1, v)
                t_r = idx(u + 1, v + 1)

                triangles.append([b_l, b_r, t_l])
                triangles.append([b_r, t_r, t_l])

        return coordinate_3d, torch.tensor(triangles)


In [13]:
nurbs = NURBS(control_points_flat.reshape(8, 5, 4), knots_u, knots_v, degree=2)
points, triangles = nurbs.tessellate()

In [14]:
points.shape

torch.Size([4500, 3])

In [15]:
all(points[..., 0]**2 + points[..., 1]**2 + points[..., 2]**2 <= 1.1)

True

In [16]:
points

tensor([[ 0.,  0., -1.],
        [ 0.,  0., -1.],
        [ 0.,  0., -1.],
        ...,
        [ 0.,  0.,  1.],
        [ 0.,  0.,  1.],
        [ 0.,  0.,  1.]])

In [17]:
x = points[:, 0].cpu().numpy()
y = points[:, 1].cpu().numpy()
z = points[:, 2].cpu().numpy()

i = triangles[:, 0].cpu().numpy()
j = triangles[:, 1].cpu().numpy()
k = triangles[:, 2].cpu().numpy()

mesh = go.Mesh3d(
    x=x, y=y, z=z,
    i=i, j=j, k=k,
    color='lightblue',
    opacity=0.5
)

fig = go.Figure(data=[mesh])
fig.show()

In [18]:
nurbs.cox_de_boor(torch.linspace(0, 1, 10), knots_u)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.6049, 0.3457, 0.0494, 0.0000, 0.0000],
        [0.3086, 0.4938, 0.1975, 0.0000, 0.0000],
        [0.1111, 0.4444, 0.4444, 0.0000, 0.0000],
        [0.0123, 0.1975, 0.7901, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.7901, 0.1975, 0.0123],
        [0.0000, 0.0000, 0.4444, 0.4444, 0.1111],
        [0.0000, 0.0000, 0.1975, 0.4938, 0.3086],
        [0.0000, 0.0000, 0.0494, 0.3457, 0.6049],
        [0.0000, 0.0000, 0.0000, 0.0000, 1.0000]])